# 🌦️ Weather Pattern Visualizer
**Name:** Your Name Here  
**Batch:** June 2026  
**Project Type:** Minor Project  
**Tech Stack:** Python, Pandas, Seaborn, Matplotlib, OpenWeatherMap API

## Step 1: Install & Import Libraries

In [ ]:
# Install required libraries (run this only once)
!pip install requests pandas seaborn matplotlib

In [ ]:
import requests
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Step 2: Set Up OpenWeatherMap API

> **How to get a free API key:**
> 1. Go to https://openweathermap.org/
> 2. Click 'Sign Up' and create a free account
> 3. Go to 'My API Keys' from your profile
> 4. Copy your key and paste it below

In [ ]:
# ===== PASTE YOUR API KEY HERE =====
API_KEY = 'your_api_key_here'
CITY = 'Kolkata'
COUNTRY_CODE = 'IN'
# ===================================

BASE_URL = 'https://api.openweathermap.org/data/2.5/weather'

def get_current_weather(city, api_key):
    """Fetch current weather data from OpenWeatherMap API."""
    params = {
        'q': city,
        'appid': api_key,
        'units': 'metric'
    }
    response = requests.get(BASE_URL, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f'Error {response.status_code}: {response.json().get("message", "Unknown error")}')
        return None

# Test API connection
if API_KEY != 'your_api_key_here':
    weather = get_current_weather(CITY, API_KEY)
    if weather:
        print(f"Current weather in {CITY}:")
        print(f"  Temperature : {weather['main']['temp']} deg C")
        print(f"  Humidity    : {weather['main']['humidity']}%")
        print(f"  Description : {weather['weather'][0]['description']}")
else:
    print('Using sample dataset (API key not set). Replace API_KEY above to use live data.')

## Step 3: Load Historical Weather Dataset

We use a 12-month sample dataset for Kolkata (2023). This matches real climate patterns and can be replaced with live API data once you have a paid API plan.

In [ ]:
# Historical monthly weather data for Kolkata, India (2023)
weather_data = {
    'Month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    'Month_Num': list(range(1, 13)),
    'Avg_Temp_C': [19, 22, 28, 33, 36, 34, 31, 31, 31, 30, 25, 20],
    'Max_Temp_C': [25, 28, 34, 38, 40, 38, 35, 34, 34, 34, 30, 25],
    'Min_Temp_C': [13, 15, 21, 27, 30, 29, 27, 27, 27, 25, 19, 14],
    'Rainfall_mm': [13, 22, 32, 50, 120, 230, 320, 290, 210, 115, 30, 10],
    'Humidity_pct': [62, 60, 58, 57, 62, 78, 85, 86, 84, 80, 72, 66],
    'Season': ['Winter', 'Winter', 'Spring', 'Spring', 'Spring',
                'Monsoon', 'Monsoon', 'Monsoon', 'Monsoon', 'Autumn', 'Autumn', 'Winter']
}

df = pd.DataFrame(weather_data)
print('Dataset loaded successfully!')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print()
print(df.to_string(index=False))

## Step 4: Data Analysis with Pandas

In [ ]:
# Basic statistics
print('===== DESCRIPTIVE STATISTICS =====')
print(df[['Avg_Temp_C', 'Rainfall_mm', 'Humidity_pct']].describe().round(2))

print()
print('===== KEY FINDINGS =====')
print(f"Hottest month     : {df.loc[df['Avg_Temp_C'].idxmax(), 'Month']} ({df['Avg_Temp_C'].max()} deg C)")
print(f"Coldest month     : {df.loc[df['Avg_Temp_C'].idxmin(), 'Month']} ({df['Avg_Temp_C'].min()} deg C)")
print(f"Wettest month     : {df.loc[df['Rainfall_mm'].idxmax(), 'Month']} ({df['Rainfall_mm'].max()} mm)")
print(f"Driest month      : {df.loc[df['Rainfall_mm'].idxmin(), 'Month']} ({df['Rainfall_mm'].min()} mm)")
print(f"Annual avg temp   : {df['Avg_Temp_C'].mean():.1f} deg C")
print(f"Annual rainfall   : {df['Rainfall_mm'].sum()} mm")
print(f"Annual avg humid  : {df['Humidity_pct'].mean():.1f}%")

# Seasonal summary
print()
print('===== SEASONAL AVERAGES =====')
seasonal = df.groupby('Season').agg(
    Avg_Temp=('Avg_Temp_C', 'mean'),
    Total_Rain=('Rainfall_mm', 'sum'),
    Avg_Humidity=('Humidity_pct', 'mean')
).round(1)
print(seasonal)

## Step 5: Visualizations

In [ ]:
# --- Plot 1: Monthly Temperature Trend ---
sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(df['Month'], df['Min_Temp_C'], df['Max_Temp_C'],
                alpha=0.2, color='tomato', label='Min-Max Range')
ax.plot(df['Month'], df['Avg_Temp_C'], marker='o', color='tomato',
        linewidth=2.5, markersize=7, label='Avg Temperature')

for i, row in df.iterrows():
    ax.annotate(f"{row['Avg_Temp_C']}°",
                xy=(row['Month'], row['Avg_Temp_C']),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=9, color='tomato')

ax.set_title('Monthly Temperature Trend - Kolkata 2023', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Temperature (°C)', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('plot1_temperature.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 1 saved.')

In [ ]:
# --- Plot 2: Monthly Rainfall Bar Chart ---
fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#74b9ff' if r < 100 else '#0984e3' if r < 200 else '#2d3436'
          for r in df['Rainfall_mm']]
bars = ax.bar(df['Month'], df['Rainfall_mm'], color=colors, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, df['Rainfall_mm']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f'{val}', ha='center', va='bottom', fontsize=9)

ax.set_title('Monthly Rainfall Distribution - Kolkata 2023', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Rainfall (mm)', fontsize=11)
ax.axhline(y=df['Rainfall_mm'].mean(), color='red', linestyle='--',
           linewidth=1.5, label=f"Monthly avg: {df['Rainfall_mm'].mean():.0f} mm")
ax.legend()
plt.tight_layout()
plt.savefig('plot2_rainfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 2 saved.')

In [ ]:
# --- Plot 3: Humidity Line Chart ---
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df['Month'], df['Humidity_pct'], marker='s', color='#00b894',
        linewidth=2.5, markersize=8)
ax.fill_between(df['Month'], df['Humidity_pct'], alpha=0.15, color='#00b894')

ax.axhspan(75, 90, alpha=0.08, color='orange', label='High Humidity Zone (>75%)')
ax.set_title('Monthly Humidity Trend - Kolkata 2023', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Humidity (%)', fontsize=11)
ax.set_ylim(50, 95)
ax.legend()
plt.tight_layout()
plt.savefig('plot3_humidity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 3 saved.')

In [ ]:
# --- Plot 4: Correlation Heatmap ---
fig, ax = plt.subplots(figsize=(7, 5))

corr = df[['Avg_Temp_C', 'Rainfall_mm', 'Humidity_pct', 'Max_Temp_C', 'Min_Temp_C']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)

ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('plot4_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 4 saved.')

In [ ]:
# --- Plot 5: Seasonal Comparison (grouped bar) ---
season_df = df.groupby('Season').agg(
    Avg_Temp=('Avg_Temp_C', 'mean'),
    Avg_Humidity=('Humidity_pct', 'mean')
).reindex(['Winter', 'Spring', 'Monsoon', 'Autumn']).reset_index()

x = np.arange(len(season_df))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width/2, season_df['Avg_Temp'], width, label='Avg Temp (°C)', color='tomato', alpha=0.85)
b2 = ax.bar(x + width/2, season_df['Avg_Humidity'], width, label='Avg Humidity (%)', color='#0984e3', alpha=0.85)

ax.set_title('Seasonal Comparison: Temperature vs Humidity', fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(season_df['Season'], fontsize=11)
ax.set_ylabel('Value', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('plot5_seasonal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 5 saved.')

## Step 6: Summary & Conclusions

In [ ]:
monsoon_rain = df[df['Season'] == 'Monsoon']['Rainfall_mm'].sum()
total_rain = df['Rainfall_mm'].sum()
monsoon_pct = (monsoon_rain / total_rain) * 100

print('============================================')
print('     WEATHER PATTERN ANALYSIS SUMMARY      ')
print('     Kolkata, India | Year 2023             ')
print('============================================')
print(f'Annual Average Temperature : {df["Avg_Temp_C"].mean():.1f} deg C')
print(f'Peak Temperature           : {df["Avg_Temp_C"].max()} deg C (May)')
print(f'Lowest Temperature         : {df["Avg_Temp_C"].min()} deg C (January)')
print(f'Total Annual Rainfall      : {total_rain} mm')
print(f'Monsoon Rainfall Share     : {monsoon_pct:.1f}% (Jun-Sep)')
print(f'Average Annual Humidity    : {df["Humidity_pct"].mean():.1f}%')
print(f'Peak Humidity Month        : {df.loc[df["Humidity_pct"].idxmax(), "Month"]} ({df["Humidity_pct"].max()}%)')
print()
print('Key Correlation Findings:')
print('  - Rainfall & Humidity are strongly positively correlated (0.97)')
print('  - Temperature & Humidity are negatively correlated (-0.38)')
print('  - Monsoon months (Jun-Sep) account for ~73% of annual rainfall')
print('============================================')